# 🔬 THE ULTIMATE DISCOVERY LAB## 240 Hours of Research → ONE Notebook → FIND THE EDGE**GOAL:** Test EVERY philosophy from the past week. Find patterns with 65%+ win rate. Or PROVE nothing works.**PHILOSOPHY:** "No saviors, just clues. Test everything. Kill what fails." - DAY4_CLUE_LOG**WHAT WE'RE TESTING:**1. Volume Precedes Price (47% of big moves had volume spike BEFORE price)2. Sector Momentum (if one sector is going, it goes for a while)3. Sympathy Plays (Leader pops → Laggard follows)4. Fire/Fuel Model (pre-positioning on known catalysts)5. Down 3 Days Reversal (contrarian bounce)6. Fade The Spike (gap up fades)7. Silent Movers (big moves on LOW volume = anomaly)8. 82.4% WIN RATE PATTERN (AI Nuclear Dip from pattern_battle_results.json)

In [ ]:
import pandas as pdimport numpy as npimport yfinance as yffrom datetime import datetime, timedeltaimport warningswarnings.filterwarnings('ignore')print("="*70)print("🔬 ULTIMATE DISCOVERY LAB - ALL RESEARCH TESTED")print("="*70)print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")print("Status: Ready to find the edge or prove nothing works")

In [ ]:
# HIGH VOLATILITY MOVERS - Tickers that ACTUALLY move (from research)HIGH_VOLATILITY_MOVERS = [    # Quantum (extreme volatility)    'IONQ', 'RGTI', 'QMCO', 'QUBT',    # Crypto Miners    'MARA', 'RIOT', 'CLSK', 'COIN',    # Space    'RKLB', 'ASTS', 'SPIR', 'LUNR',    # Biotech    'NTLA', 'BEAM', 'CRSP', 'RXRX', 'AKRO', 'VKTX', 'MRNA',    # AI/Tech    'NVDA', 'AMD', 'PLTR', 'SMCI', 'AI', 'PATH', 'SNOW',    # AVs & EVs    'KDK', 'ACHR', 'JOBY', 'TSLA', 'RIVN', 'LCID', 'QS',    # Clean Energy    'PLUG', 'FCEL', 'ENPH', 'RUN',    # Fintech    'SOFI', 'UPST', 'AFRM', 'HOOD', 'SQ',    # Small Cap Movers    'CELH', 'ELF', 'DUOL', 'ONON', 'APP']# SECTOR GROUPINGS (for sympathy play testing)SECTOR_GROUPS = {    'quantum': ['IONQ', 'RGTI', 'QMCO', 'QUBT'],    'crypto': ['MARA', 'RIOT', 'CLSK', 'COIN'],    'space': ['RKLB', 'ASTS', 'SPIR', 'LUNR'],    'biotech': ['NTLA', 'BEAM', 'CRSP', 'RXRX', 'AKRO', 'VKTX', 'MRNA'],    'ai_chips': ['NVDA', 'AMD', 'SMCI'],    'ev': ['TSLA', 'RIVN', 'LCID', 'QS'],    'clean_energy': ['PLUG', 'FCEL', 'ENPH', 'RUN'],    'fintech': ['SOFI', 'UPST', 'AFRM', 'HOOD', 'SQ']}print(f"Universe: {len(HIGH_VOLATILITY_MOVERS)} tickers")print(f"Sectors: {len(SECTOR_GROUPS)}")

In [ ]:
def get_data(ticker, days_back=180):    """Fetch OHLCV data with caching"""    try:        end = datetime.now()        start = end - timedelta(days=days_back)        df = yf.download(ticker, start=start, end=end, progress=False)        if len(df) > 0:            return df    except:        pass    return None# Testtest = get_data('IONQ', days_back=30)if test is not None:    print(f"✅ Data fetcher working: {len(test)} bars for IONQ")else:    print("❌ Data fetcher failed")

## 🔬 TEST 1: VOLUME PRECEDES PRICE**From DAY4_CLUE_LOG:** "47% of big moves had 2x+ volume spike 1-3 days BEFORE price moved"**Hypothesis:** Volume spikes predict price moves. Buy when volume spikes but price hasn't moved yet.

In [ ]:
def test_volume_precedes_price():    """    TEST: Does volume spike 1-3 days BEFORE big price moves?    Entry: Volume > 2x average, price change < 3%    Exit: Next 1-3 days    WIN: Price moves >5% after entry    """    print("\n" + "="*70)    print("🔬 TEST 1: VOLUME PRECEDES PRICE")    print("="*70)        all_signals = []        for ticker in HIGH_VOLATILITY_MOVERS:        data = get_data(ticker, days_back=180)        if data is None or len(data) < 30:            continue                close = data['Close'].values        volume = data['Volume'].values        vol_ma = pd.Series(volume).rolling(20).mean().values                for i in range(25, len(data) - 5):            # Entry: Volume spike but price hasn't moved            vol_ratio = volume[i] / vol_ma[i] if vol_ma[i] > 0 else 0            price_change = (close[i] / close[i-1] - 1) * 100                        if vol_ratio > 2.0 and abs(price_change) < 3:                # Check next 3 days                max_gain = max((close[i+j] / close[i] - 1) * 100 for j in range(1, min(4, len(data)-i)))                max_loss = min((close[i+j] / close[i] - 1) * 100 for j in range(1, min(4, len(data)-i)))                                win = max_gain > 5                all_signals.append({                    'ticker': ticker,                    'vol_ratio': vol_ratio,                    'max_gain': max_gain,                    'max_loss': max_loss,                    'win': win                })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_gain = df['max_gain'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Max Gain: {avg_gain:.1f}%")        print(f"   Status: {'✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonevol_results = test_volume_precedes_price()

## 🔬 TEST 2: SECTOR MOMENTUM**From SECTOR_MOMENTUM_HYPOTHESIS.md:** "If one sector is going, it goes for a while"**Hypothesis:** When 3+ stocks in a sector move >3%, the whole sector continues.

In [ ]:
def test_sector_momentum():    """    TEST: Does sector-wide momentum continue?    Entry: 3+ stocks in sector up >3% today    Exit: Hold 1-3 days    WIN: Sector continues up >2%    """    print("\n" + "="*70)    print("🔬 TEST 2: SECTOR MOMENTUM")    print("="*70)        all_signals = []        for sector_name, tickers in SECTOR_GROUPS.items():        # Get data for all tickers in sector        sector_data = {}        for ticker in tickers:            data = get_data(ticker, days_back=180)            if data is not None and len(data) > 30:                sector_data[ticker] = data                if len(sector_data) < 3:            continue                # Find days where 3+ stocks moved >3%        min_days = min(len(d) for d in sector_data.values())                for i in range(5, min_days - 5):            movers = 0            for ticker, data in sector_data.items():                if len(data) > i:                    daily_change = (data['Close'].iloc[i] / data['Close'].iloc[i-1] - 1) * 100                    if daily_change > 3:                        movers += 1                        if movers >= 3:                # Check next 3 days sector performance                next_gains = []                for ticker, data in sector_data.items():                    if len(data) > i + 3:                        gain = (data['Close'].iloc[i+3] / data['Close'].iloc[i] - 1) * 100                        next_gains.append(gain)                                if len(next_gains) > 0:                    avg_gain = np.mean(next_gains)                    win = avg_gain > 2                    all_signals.append({                        'sector': sector_name,                        'movers': movers,                        'avg_gain': avg_gain,                        'win': win                    })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_gain = df['avg_gain'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Gain: {avg_gain:.1f}%")        print(f"   Status: {'✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonesector_results = test_sector_momentum()

## 🔬 TEST 3: SYMPATHY PLAYS (Leader/Laggard)**From SYMPATHY_PLAY_STRATEGY.md:** "When a Leader pops, buy the Laggards"**Hypothesis:** When sector leader moves >10%, laggards catch up in 1-3 days.

In [ ]:
def test_sympathy_plays():    """    TEST: Do laggards follow leaders?    Entry: Leader up >10%, laggard flat/down    Exit: 1-3 days    WIN: Laggard moves >5%    """    print("\n" + "="*70)    print("🔬 TEST 3: SYMPATHY PLAYS")    print("="*70)        all_signals = []        for sector_name, tickers in SECTOR_GROUPS.items():        if len(tickers) < 2:            continue                sector_data = {}        for ticker in tickers:            data = get_data(ticker, days_back=180)            if data is not None and len(data) > 30:                sector_data[ticker] = data                if len(sector_data) < 2:            continue                min_days = min(len(d) for d in sector_data.values())                for i in range(5, min_days - 5):            # Find leader (biggest mover)            daily_changes = {}            for ticker, data in sector_data.items():                change = (data['Close'].iloc[i] / data['Close'].iloc[i-1] - 1) * 100                daily_changes[ticker] = change                        leader = max(daily_changes, key=daily_changes.get)            leader_change = daily_changes[leader]                        if leader_change > 10:                # Check laggards                for ticker in sector_data:                    if ticker != leader and daily_changes[ticker] < 2:                        data = sector_data[ticker]                        if len(data) > i + 3:                            # Check if laggard catches up                            max_gain = max((data['Close'].iloc[i+j] / data['Close'].iloc[i] - 1) * 100                                           for j in range(1, 4))                            win = max_gain > 5                            all_signals.append({                                'sector': sector_name,                                'leader': leader,                                'laggard': ticker,                                'leader_move': leader_change,                                'laggard_gain': max_gain,                                'win': win                            })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_gain = df['laggard_gain'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Laggard Gain: {avg_gain:.1f}%")        print(f"   Status: {'✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonesympathy_results = test_sympathy_plays()

## 🔬 TEST 4: DOWN 3 DAYS REVERSAL**Contrarian hypothesis:** After 3 consecutive down days, bounce probability increases.

In [ ]:
def test_down_3_days():    """    TEST: Buy after 3 consecutive down days    Entry: 3 red days in a row    Exit: Next 1-3 days    WIN: Bounce >3%    """    print("\n" + "="*70)    print("🔬 TEST 4: DOWN 3 DAYS REVERSAL")    print("="*70)        all_signals = []        for ticker in HIGH_VOLATILITY_MOVERS:        data = get_data(ticker, days_back=180)        if data is None or len(data) < 30:            continue                close = data['Close'].values                for i in range(5, len(data) - 5):            # Check for 3 consecutive down days            down1 = close[i] < close[i-1]            down2 = close[i-1] < close[i-2]            down3 = close[i-2] < close[i-3]                        total_drop = (close[i] / close[i-3] - 1) * 100                        if down1 and down2 and down3 and total_drop < -5:                # Check bounce                max_gain = max((close[i+j] / close[i] - 1) * 100 for j in range(1, min(4, len(data)-i)))                win = max_gain > 3                                all_signals.append({                    'ticker': ticker,                    'total_drop': total_drop,                    'bounce': max_gain,                    'win': win                })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_bounce = df['bounce'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Bounce: {avg_bounce:.1f}%")        print(f"   Status: {'✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonedown3_results = test_down_3_days()

## 🔬 TEST 5: FADE THE SPIKE**Hypothesis:** Gap up >5% fades - short the spike.

In [ ]:
def test_fade_spike():    """    TEST: Do gap-ups fade?    Entry: Gap up >5%    Exit: End of day or next day    WIN: Price fades >2% from high    """    print("\n" + "="*70)    print("🔬 TEST 5: FADE THE SPIKE")    print("="*70)        all_signals = []        for ticker in HIGH_VOLATILITY_MOVERS:        data = get_data(ticker, days_back=180)        if data is None or len(data) < 30:            continue                for i in range(2, len(data) - 2):            # Gap up >5%            gap = (data['Open'].iloc[i] / data['Close'].iloc[i-1] - 1) * 100                        if gap > 5:                # Check if it fades                high = data['High'].iloc[i]                close = data['Close'].iloc[i]                fade = (high - close) / high * 100                                win = fade > 2                all_signals.append({                    'ticker': ticker,                    'gap': gap,                    'fade': fade,                    'win': win                })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_fade = df['fade'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Fade: {avg_fade:.1f}%")        print(f"   Status: {'✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonefade_results = test_fade_spike()

## 🔬 TEST 6: AI NUCLEAR DIP (82.4% WIN RATE FROM GOLD_FOUND)**From GOLD_FOUND_ANALYSIS.md:** RSI < 21, bounce > 8%**This is the HIGHEST WIN RATE pattern found in the entire codebase!**

In [ ]:
def test_nuclear_dip():    """    TEST: The 82.4% win rate pattern    Entry: RSI < 21 (extreme oversold)    Exit: Wait for 8%+ bounce    WIN: Price bounces >8%    """    print("\n" + "="*70)    print("🔬 TEST 6: AI NUCLEAR DIP (82.4% historical)")    print("="*70)        all_signals = []        for ticker in HIGH_VOLATILITY_MOVERS:        data = get_data(ticker, days_back=180)        if data is None or len(data) < 30:            continue                close = data['Close']                # Calculate RSI        delta = close.diff()        gain = delta.where(delta > 0, 0)        loss = -delta.where(delta < 0, 0)        avg_gain = gain.rolling(14).mean()        avg_loss = loss.rolling(14).mean()        rs = avg_gain / (avg_loss + 0.0001)        rsi = 100 - (100 / (1 + rs))                for i in range(20, len(data) - 10):            if rsi.iloc[i] < 21:  # Extreme oversold                # Check for bounce in next 10 days                entry = close.iloc[i]                max_gain = max((close.iloc[i+j] / entry - 1) * 100 for j in range(1, min(11, len(data)-i)))                                win = max_gain > 8                all_signals.append({                    'ticker': ticker,                    'rsi': rsi.iloc[i],                    'max_bounce': max_gain,                    'win': win                })        if len(all_signals) > 0:        df = pd.DataFrame(all_signals)        win_rate = df['win'].mean() * 100        avg_bounce = df['max_bounce'].mean()                print(f"\n📊 RESULTS:")        print(f"   Signals found: {len(df)}")        print(f"   WIN RATE: {win_rate:.1f}%")        print(f"   Avg Bounce: {avg_bounce:.1f}%")        print(f"   Status: {'🔥 WINNER!' if win_rate > 65 else '✅ PROMISING' if win_rate > 55 else '❌ NOT WORKING'}")        return df    else:        print("❌ No signals found")        return Nonenuclear_results = test_nuclear_dip()

## 📊 FINAL VERDICT: WHAT WORKS?

In [ ]:
print("\n" + "="*70)print("📊 FINAL VERDICT - WHAT WORKS?")print("="*70)results = {    'Volume Precedes Price': vol_results,    'Sector Momentum': sector_results,    'Sympathy Plays': sympathy_results,    'Down 3 Days': down3_results,    'Fade Spike': fade_results,    'Nuclear Dip (RSI<21)': nuclear_results}print("\n" + "-"*70)print(f"{'TEST':<25} {'SIGNALS':<10} {'WIN RATE':<12} {'STATUS'}")print("-"*70)winners = []for name, df in results.items():    if df is not None and len(df) > 0:        win_rate = df['win'].mean() * 100        signals = len(df)        status = '🔥 WINNER' if win_rate >= 65 else '✅ PROMISING' if win_rate >= 55 else '❌ FAIL'        print(f"{name:<25} {signals:<10} {win_rate:.1f}%{'':<6} {status}")        if win_rate >= 65:            winners.append(name)    else:        print(f"{name:<25} {'0':<10} {'N/A':<12} ❌ NO DATA")print("\n" + "="*70)if len(winners) > 0:    print(f"🏆 WINNERS (65%+ win rate): {', '.join(winners)}")    print("\n🎯 NEXT STEP: Build live scanner for winning patterns!")else:    print("❌ NO PATTERNS ABOVE 65% - NEED TO PIVOT")    print("\n🎯 NEXT STEP: Test different parameters or find new hypotheses")print("="*70)